problem 1

In [10]:
import collections

class Matrix:

    def __init__(self, *args):
        self._data = []
        self._rows = 0
        self._cols = 0

        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):

            n, m = args
            if n <= 0 or m <= 0:
                raise ValueError("Matrix dimensions must be positive integers.")
            self._rows = n
            self._cols = m
            self._data = [[0] * m for _ in range(n)]
        elif len(args) == 1 and isinstance(args[0], (list, tuple)):

            list_of_lists = args[0]
            self._set_data_from_list(list_of_lists)
        else:
            raise TypeError("Matrix can be initialized with (rows, cols) or a list of lists.")

    def _set_data_from_list(self, list_of_lists):
        if not list_of_lists:
            raise ValueError("Cannot initialize Matrix with an empty list of lists.")

        first_row_len = len(list_of_lists[0])
        if first_row_len == 0:
            raise ValueError("Cannot initialize Matrix with rows containing no columns.")

        if not all(len(row) == first_row_len for row in list_of_lists):
            raise ValueError("All rows in the list of lists must have the same number of columns.")

        self._rows = len(list_of_lists)
        self._cols = first_row_len
        self._data = [row[:] for row in list_of_lists]

    @property
    def rows(self):

        return self._rows

    @property
    def cols(self):

        return self._cols

    def __getitem__(self, key):

        if isinstance(key, tuple):
            row_idx_or_slice, col_idx_or_slice = key

            if isinstance(row_idx_or_slice, int) and isinstance(col_idx_or_slice, int):
                if not (0 <= row_idx_or_slice < self._rows and 0 <= col_idx_or_slice < self._cols):
                    raise IndexError(f"Matrix index ({row_idx_or_slice},{col_idx_or_slice}) out of bounds for matrix size ({self._rows},{self._cols}).")
                return self._data[row_idx_or_slice][col_idx_or_slice]

            else:

                if isinstance(row_idx_or_slice, int):
                    if not (0 <= row_idx_or_slice < self._rows):
                        raise IndexError(f"Row index {row_idx_or_slice} out of bounds for matrix with {self._rows} rows.")
                    selected_rows_indices = [row_idx_or_slice]
                elif isinstance(row_idx_or_slice, slice):
                    start, stop, step = row_idx_or_slice.indices(self._rows)
                    selected_rows_indices = list(range(start, stop, step))
                else:
                    raise TypeError("Row index must be an integer or a slice.")


                if isinstance(col_idx_or_slice, int):
                    if not (0 <= col_idx_or_slice < self._cols):
                        raise IndexError(f"Column index {col_idx_or_slice} out of bounds for matrix with {self._cols} columns.")
                    new_data = [[self._data[r_idx][col_idx_or_slice]] for r_idx in selected_rows_indices]
                elif isinstance(col_idx_or_slice, slice):
                    start, stop, step = col_idx_or_slice.indices(self._cols)
                    new_data = [
                        [self._data[r_idx][c_idx] for c_idx in range(start, stop, step)]
                        for r_idx in selected_rows_indices
                    ]
                else:
                    raise TypeError("Column index must be an integer or a slice.")

                return Matrix(new_data)
        elif isinstance(key, int):
            if not (0 <= key < self._rows):
                raise IndexError(f"Row index {key} out of bounds for matrix with {self._rows} rows.")
            return self._MatrixRow(self, key)
        elif isinstance(key, slice):
            start, stop, step = key.indices(self._rows)
            new_data = [row[:] for row in self._data[start:stop:step]]
            return Matrix(new_data)
        else:
            raise TypeError("Matrix index must be an integer, a slice, or a (row, col) tuple/slice.")

    def __setitem__(self, key, value):

        if isinstance(key, tuple):

            if len(key) != 2:
                raise IndexError("Matrix index must be a single integer or a (row, col) tuple.")
            i, j = key
            if not (0 <= i < self._rows and 0 <= j < self._cols):
                raise IndexError(f"Matrix index ({i},{j}) out of bounds for matrix size ({self._rows},{self._cols}).")
            self._data[i][j] = value
        else:
            raise TypeError("Direct assignment to a row (e.g., M[i] = ...) is not supported. Use M[i,j] = value or M[i][j] = value.")

    class _MatrixRow(collections.abc.MutableSequence):

        def __init__(self, matrix, row_index):
            self._matrix = matrix
            self._row_index = row_index

        def __len__(self):
            return self._matrix.cols

        def __getitem__(self, col_index):
            return self._matrix[self._row_index, col_index]

        def __setitem__(self, col_index, value):
            self._matrix[self._row_index, col_index] = value

        def __delitem__(self, col_index):
            raise TypeError("Deletion of matrix elements is not supported.")

        def insert(self, index, value):
            raise TypeError("Insertion into matrix rows is not supported.")

        def __repr__(self):
            return repr(self._matrix._data[self._row_index])

        def __str__(self):
            return ' '.join(map(str, self._matrix._data[self._row_index]))

    def assign(self, other):

        if isinstance(other, Matrix):
            if self.rows != other.rows or self.cols != other.cols:
                raise ValueError(f"Cannot assign matrices of different sizes. Expected ({self.rows}, {self.cols}), got ({other.rows}, {other.cols}).")

            for r in range(self.rows):
                for c in range(self.cols):
                    self._data[r][c] = other[r, c]
        elif isinstance(other, (list, tuple)):

            if not other:
                raise ValueError("Cannot assign an empty list of lists.")

            new_rows = len(other)
            if not other[0]:
                 raise ValueError("Cannot assign a list of lists with empty rows.")
            new_cols = len(other[0])

            if not all(len(row) == new_cols for row in other):
                raise ValueError("All rows in the assigned list of lists must have the same number of columns.")

            if self.rows != new_rows or self.cols != new_cols:
                raise ValueError(f"Cannot assign list of lists of different size. Expected ({self.rows}, {self.cols}), got ({new_rows}, {new_cols}).")

            for r in range(self.rows):
                for c in range(self.cols):
                    self._data[r][c] = other[r][c]
        else:
            raise TypeError("Cannot assign from an object that is not a Matrix or a list of lists.")

    def shape(self):

        return (self._rows, self._cols)

    def transpose(self):

        transposed_data = [[self._data[r][c] for r in range(self._rows)] for c in range(self._cols)]
        return Matrix(transposed_data)

    def row(self, n):
        if not (0 <= n < self._rows):
            raise IndexError(f"Row index {n} out of bounds for matrix with {self._rows} rows.")
        return Matrix([self._data[n]])

    def column(self, n):

        if not (0 <= n < self._cols):
            raise IndexError(f"Column index {n} out of bounds for matrix with {self._cols} columns.")
        col_data = [[self._data[r][n]] for r in range(self._rows)]
        return Matrix(col_data)

    def to_list(self):

        return [row[:] for row in self._data]
    def block(self, r_start, r_end, c_start, c_end):

        if not (0 <= r_start < self._rows and 0 < r_end <= self._rows and r_start < r_end):
            raise ValueError(f"Invalid row range ({r_start}, {r_end}) for matrix with {self._rows} rows.")
        if not (0 <= c_start < self._cols and 0 < c_end <= self._cols and c_start < c_end):
            raise ValueError(f"Invalid column range ({c_start}, {c_end}) for matrix with {self._cols} columns.")

        new_data = []
        for r in range(r_start, r_end):
            new_data.append(self._data[r][c_start:c_end])
        return Matrix(new_data)
# problem 4 added
    def scalarmul(self, c):

        new_data = [[element * c for element in row] for row in self._data]
        return Matrix(new_data)

    def add(self, N):

        if self.shape() != N.shape():
            raise ValueError(f"Matrix addition requires matrices of the same dimensions. Got {self.shape()} and {N.shape()}.")
        new_data = [
            [self._data[r][c] + N[r, c] for c in range(self._cols)]
            for r in range(self._rows)
        ]
        return Matrix(new_data)

    def __add__(self, other):
        if isinstance(other, Matrix):
            return self.add(other)
        return NotImplemented

    def sub(self, N):

        if self.shape() != N.shape():
            raise ValueError(f"Matrix subtraction requires matrices of the same dimensions. Got {self.shape()} and {N.shape()}.")
        new_data = [
            [self._data[r][c] - N[r, c] for c in range(self._cols)]
            for r in range(self._rows)
        ]
        return Matrix(new_data)

    def __sub__(self, other):
        if isinstance(other, Matrix):
            return self.sub(other)
        return NotImplemented

    def mat_mult(self, N):

        if self.cols != N.rows:
            raise ValueError(f"Matrix multiplication requires self.cols ({self.cols}) to equal N.rows ({N.rows}).")

        new_data = [[0 for _ in range(N.cols)] for _ in range(self._rows)]
        for r in range(self._rows):
            for c in range(N.cols):
                for k in range(self._cols):
                    new_data[r][c] += self._data[r][k] * N[k, c]
        return Matrix(new_data)

    def __matmul__(self, other):
        if isinstance(other, Matrix):
            return self.mat_mult(other)
        return NotImplemented

    def element_mult(self, N):

        if self.shape() != N.shape():
            raise ValueError(f"Element-wise multiplication requires matrices of the same dimensions. Got {self.shape()} and {N.shape()}.")
        new_data = [
            [self._data[r][c] * N[r, c] for c in range(self._cols)]
            for r in range(self._rows)
        ]
        return Matrix(new_data)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, Matrix):

            return self.element_mult(other)
        return NotImplemented

    def __rmul__(self, other):

        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        return NotImplemented

    def __eq__(self, other):

        if not isinstance(other, Matrix):
            return NotImplemented
        if self.rows != other.rows or self.cols != other.cols:
            return False
        for r in range(self.rows):
            for c in range(self.cols):
                if self._data[r][c] != other[r, c]:
                    return False
        return True

    def __repr__(self):
        return f"Matrix(rows={self._rows}, cols={self._cols}, data={self._data})"

    def __str__(self):
        rows_str = []
        for row in self._data:
            rows_str.append(' '.join(map(str, row)))
        return '\n'.join(rows_str)

In [2]:
print("--- Demo 1: Initialize with n, m ---")
try:
    m1 = Matrix(2, 3)
    print("Matrix m1 (2x3, zeros):")
    print(m1)
    print(f"m1.rows: {m1.rows}, m1.cols: {m1.cols}")

    print("\nTrying to initialize with invalid dimensions (0 rows):")
    m_invalid_dim = Matrix(0, 3)
except ValueError as e:
    print(f"Error initializing with invalid dimensions: {e}")
print("\n")


print("--- Demo 2: Initialize with list of lists ---")
try:
    data = [[1, 2, 3], [4, 5, 6]]
    m2 = Matrix(data)
    print("Matrix m2 (from [[1, 2, 3], [4, 5, 6]]):")
    print(m2)
    print(f"m2.rows: {m2.rows}, m2.cols: {m2.cols}")

    print("\nTrying to initialize with invalid list of lists (ragged array):")
    invalid_data = [[1, 2], [3, 4, 5]]
    m_ragged = Matrix(invalid_data)
except ValueError as e:
    print(f"Error initializing with ragged list of lists: {e}")

try:
    print("\nTrying to initialize with empty list of lists:")
    m_empty_list = Matrix([])
except ValueError as e:
    print(f"Error initializing with empty list of lists: {e}")

try:
    print("\nTrying to initialize with list of lists containing empty rows:")
    m_empty_row = Matrix([[]])
except ValueError as e:
    print(f"Error initializing with list of lists containing empty rows: {e}")
print("\n")

print("--- Demo 3: Indexing ---")
m3 = Matrix([[10, 20, 30], [40, 50, 60], [70, 80, 90]])
print("Matrix m3:")
print(m3)
print(f"m3[0][0]: {m3[0][0]}")
print(f"m3[1,2]: {m3[1,2]}")

try:
    print("\nTrying to index out of bounds (m3[3,0]):")
    print(f"m3[3,0]: {m3[3,0]}")
except IndexError as e:
    print(f"Error indexing out of bounds: {e}")
print("\n")

print("--- Demo 4: Element Assignment ---")
m4 = Matrix(3, 3)
print("Matrix m4 (initial, 3x3 zeros):")
print(m4)

m4[0, 0] = 11
m4[1][1] = 22
m4[2, 2] = 33
print("\nMatrix m4 after element assignments:")
print(m4)

try:
    print("\nTrying to assign out of bounds (m4[3, 0] = 99):")
    m4[3, 0] = 99
except IndexError as e:
    print(f"Error assigning out of bounds: {e}")
print("\n")


print("--- Demo 5: Matrix Assignment (using assign method) ---")
m5_original = Matrix(2, 2)
m5_original[0,0] = 1
m5_original[0,1] = 2
m5_original[1,0] = 3
m5_original[1,1] = 4
print("Matrix m5_original:")
print(m5_original)

m5_target = Matrix(2, 2)
print("\nMatrix m5_target (initial, 2x2 zeros):")
print(m5_target)

m5_target.assign(m5_original)
print("\nMatrix m5_target after m5_target.assign(m5_original):")
print(m5_target)
print(f"Are m5_target and m5_original equal (values)? {m5_target == m5_original}")

new_values = [[7, 8], [9, 10]]
m5_target.assign(new_values)
print("\nMatrix m5_target after m5_target.assign([[7, 8], [9, 10]]):")
print(m5_target)

print("\nTrying to assign matrices of different sizes:")
m5_smaller = Matrix(1, 1)
try:
    m5_target.assign(m5_smaller)
except ValueError as e:
    print(f"Error assigning matrices of different sizes: {e}")


print("\nTrying to assign list of lists of different sizes:")
wrong_size_list = [[1, 2, 3], [4, 5, 6]]
try:
    m5_target.assign(wrong_size_list)
except ValueError as e:
    print(f"Error assigning list of lists of different sizes: {e}")
print("\n")

print("--- Demo 6: Equality Check ---")
m_a = Matrix([[1, 2], [3, 4]])
m_b = Matrix([[1, 2], [3, 4]])
m_c = Matrix([[5, 6], [7, 8]])
m_d = Matrix(2, 3)

print(f"m_a:\n{m_a}")
print(f"m_b:\n{m_b}")
print(f"m_c:\n{m_c}")
print(f"m_d:\n{m_d}")

print(f"\nm_a == m_b: {m_a == m_b}")
print(f"m_a == m_c: {m_a == m_c}")
print(f"m_a == m_d: {m_a == m_d}")


--- Demo 1: Initialize with n, m ---
Matrix m1 (2x3, zeros):
0 0 0
0 0 0
m1.rows: 2, m1.cols: 3

Trying to initialize with invalid dimensions (0 rows):
Error initializing with invalid dimensions: Matrix dimensions must be positive integers.


--- Demo 2: Initialize with list of lists ---
Matrix m2 (from [[1, 2, 3], [4, 5, 6]]):
1 2 3
4 5 6
m2.rows: 2, m2.cols: 3

Trying to initialize with invalid list of lists (ragged array):
Error initializing with ragged list of lists: All rows in the list of lists must have the same number of columns.

Trying to initialize with empty list of lists:
Error initializing with empty list of lists: Cannot initialize Matrix with an empty list of lists.

Trying to initialize with list of lists containing empty rows:
Error initializing with list of lists containing empty rows: Cannot initialize Matrix with rows containing no columns.


--- Demo 3: Indexing ---
Matrix m3:
10 20 30
40 50 60
70 80 90
m3[0][0]: 10
m3[1,2]: 60

Trying to index out of bounds (m3[3

In [12]:
def constant(n, m, c):
    if not (isinstance(n, int) and isinstance(m, int) and n > 0 and m > 0):
        raise ValueError("n and m must be positive integers.")
    return Matrix([[float(c)] * m for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    if not (isinstance(n, int) and n > 0):
        raise ValueError("n must be a positive integer for identity matrix.")
    identity_data = [[0.0] * n for _ in range(n)]
    for i in range(n):
        identity_data[i][i] = 1.0
    return Matrix(identity_data)


problem 2

In [4]:

m_sample = Matrix([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
])
print("--- Sample Matrix for Demos ---")
print(m_sample)
print("\n")

print("--- Demo 7: shape() ---")
print(f"Shape of m_sample: {m_sample.shape()}")
print("\n")

print("--- Demo 8: transpose() ---")
m_transposed = m_sample.transpose()
print("Transposed matrix:")
print(m_transposed)
print(f"Shape of transposed matrix: {m_transposed.shape()}")
print("\n")

print("--- Demo 9: row(n) and column(n) ---")
row_1 = m_sample.row(1)
print("Row 1 (as Matrix):")
print(row_1)
print(f"Shape of row_1: {row_1.shape()}")

col_2 = m_sample.column(2)
print("\nColumn 2 (as Matrix):")
print(col_2)
print(f"Shape of col_2: {col_2.shape()}")

print("\nTrying to get out-of-bounds row (m_sample.row(9)):")
try:
    m_sample.row(9)
except IndexError as e:
    print(f"Error getting out-of-bounds row: {e}")
print("\n")

print("--- Demo 10: to_list() ---")
list_representation = m_sample.to_list()
print(f"List representation of m_sample: {list_representation}")

list_representation[0][0] = 999
print(f"Original matrix after modifying list representation: {m_sample[0,0]}")
print("\n")


print("--- Demo 11: block() ---")
block_matrix = m_sample.block(0, 2, 1, 3)
print("Block (rows 0-1, cols 1-2):")
print(block_matrix)
print(f"Shape of block_matrix: {block_matrix.shape()}")

print("\nTrying to get invalid block range (r_start >= r_end):")
try:
    m_sample.block(1, 1, 0, 2)
except ValueError as e:
    print(f"Error getting invalid block: {e}")

print("\nTrying to get out-of-bounds block range:")
try:
    m_sample.block(0, 4, 0, 2)
except ValueError as e:
    print(f"Error getting out-of-bounds block: {e}")
print("\n")

print("--- Demo 12: __getitem__ with Slicing ---")

print("Original matrix:")
print(m_sample)

slice_rows = m_sample[0:2]
print("\nm_sample[0:2] (rows 0 and 1):")
print(slice_rows)

slice_row_cols = m_sample[1, 1:4]
print("\nm_sample[1, 1:4] (row 1, cols 1 to 3, as a 1x3 matrix):")
print(slice_row_cols)

slice_col_rows = m_sample[0:3, 2]
print("\nm_sample[0:3, 2] (cols 2, rows 0 to 2, as a 3x1 matrix):")
print(slice_col_rows)

slice_block = m_sample[0:2, 1:3]
print("\nm_sample[0:2, 1:3] (rows 0 to 1, cols 1 to 2):")
print(slice_block)

all_rows = m_sample[:]
print("\nm_sample[:] (all rows):")
print(all_rows)

full_matrix_slice = m_sample[:, :]
print("\nm_sample[:, :] (entire matrix):")
print(full_matrix_slice)

all_rows_cols_1_3 = m_sample[:, 1:4]
print("\nm_sample[:, 1:4] (all rows, cols 1 to 3):")
print(all_rows_cols_1_3)

print("\nTrying invalid slice (out of bounds for row slice):")
try:
    m_sample[0:5, :]
except IndexError as e:
    print(f"Error with out-of-bounds row slice: {e}")


--- Sample Matrix for Demos ---
1 2 3 4
5 6 7 8
9 10 11 12


--- Demo 7: shape() ---
Shape of m_sample: (3, 4)


--- Demo 8: transpose() ---
Transposed matrix:
1 5 9
2 6 10
3 7 11
4 8 12
Shape of transposed matrix: (4, 3)


--- Demo 9: row(n) and column(n) ---
Row 1 (as Matrix):
5 6 7 8
Shape of row_1: (1, 4)

Column 2 (as Matrix):
3
7
11
Shape of col_2: (3, 1)

Trying to get out-of-bounds row (m_sample.row(9)):
Error getting out-of-bounds row: Row index 9 out of bounds for matrix with 3 rows.


--- Demo 10: to_list() ---
List representation of m_sample: [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
Original matrix after modifying list representation: 1


--- Demo 11: block() ---
Block (rows 0-1, cols 1-2):
2 3
6 7
Shape of block_matrix: (2, 2)

Trying to get invalid block range (r_start >= r_end):
Error getting invalid block: Invalid row range (1, 1) for matrix with 3 rows.

Trying to get out-of-bounds block range:
Error getting out-of-bounds block: Invalid row range (0, 4) for matrix

problem 3

In [6]:
print("--- Demo 13: constant(n, m, c) ---")
m_const = constant(2, 3, 7)
print("Constant matrix (2x3, filled with 7):")
print(m_const)
print("\n")

print("--- Demo 14: zeros(n, m) ---")
m_zeros = zeros(3, 2)
print("Zeros matrix (3x2):")
print(m_zeros)
print("\n")

print("--- Demo 15: ones(n, m) ---")
m_ones = ones(2, 4)
print("Ones matrix (2x4):")
print(m_ones)
print("\n")

print("--- Demo 16: eye(n) ---")
m_eye = eye(4)
print("Identity matrix (4x4):")
print(m_eye)
print("\n")

print("--- Demo 17: Error handling for special matrix functions ---")
try:
    constant(0, 5, 10)
except ValueError as e:
    print(f"Error with constant(0, 5, 10): {e}")

try:
    eye(-1)
except ValueError as e:
    print(f"Error with eye(-1): {e}")


--- Demo 13: constant(n, m, c) ---
Constant matrix (2x3, filled with 7):
7.0 7.0 7.0
7.0 7.0 7.0


--- Demo 14: zeros(n, m) ---
Zeros matrix (3x2):
0.0 0.0
0.0 0.0
0.0 0.0


--- Demo 15: ones(n, m) ---
Ones matrix (2x4):
1.0 1.0 1.0 1.0
1.0 1.0 1.0 1.0


--- Demo 16: eye(n) ---
Identity matrix (4x4):
1.0 0.0 0.0 0.0
0.0 1.0 0.0 0.0
0.0 0.0 1.0 0.0
0.0 0.0 0.0 1.0


--- Demo 17: Error handling for special matrix functions ---
Error with constant(0, 5, 10): n and m must be positive integers.
Error with eye(-1): n must be a positive integer for identity matrix.


problem 4 tests

In [9]:
print("--- Demo: scalarmul(c) ---")
m_arith = Matrix([[1, 2], [3, 4]])
print("Original Matrix m_arith:")
print(m_arith)
m_scaled = m_arith.scalarmul(2)
print("\nm_arith.scalarmul(2):")
print(m_scaled)
print("\n")

print("--- Demo: add(N) ---")
m_add1 = Matrix([[1, 1], [1, 1]])
m_add2 = Matrix([[2, 2], [2, 2]])
print("Matrix m_add1:")
print(m_add1)
print("\nMatrix m_add2:")
print(m_add2)
print("\nm_add1.add(m_add2):")
print(m_add1.add(m_add2))

print("--- Demo: sub(N) ---")
m_sub1 = Matrix([[5, 6], [7, 8]])
m_sub2 = Matrix([[1, 1], [1, 1]])
print("Matrix m_sub1:")
print(m_sub1)
print("\nMatrix m_sub2:")
print(m_sub2)
print("\nm_sub1.sub(m_sub2):")
print(m_sub1.sub(m_sub2))

print("--- Demo: mat_mult(N) (Matrix Multiplication) ---")
m_mult_A = Matrix([[1, 2], [3, 4]])
m_mult_B = Matrix([[5, 6], [7, 8]])
print("Matrix m_mult_A:")
print(m_mult_A)
print("\nMatrix m_mult_B:")
print(m_mult_B)
print("\nm_mult_A.mat_mult(m_mult_B):")
print(m_mult_A.mat_mult(m_mult_B))

m_mult_C = Matrix([[1, 2, 3]])
m_mult_D = Matrix([[4], [5], [6]])
print("\nMatrix m_mult_C:")
print(m_mult_C)
print("\nMatrix m_mult_D:")
print(m_mult_D)
print("\nm_mult_C.mat_mult(m_mult_D):")
print(m_mult_C.mat_mult(m_mult_D))

print("--- Demo: element_mult(N) ---")
m_elem_mult1 = Matrix([[1, 2], [3, 4]])
m_elem_mult2 = Matrix([[10, 20], [30, 40]])
print("Matrix m_elem_mult1:")
print(m_elem_mult1)
print("\nMatrix m_elem_mult2:")
print(m_elem_mult2)
print("\nm_elem_mult1.element_mult(m_elem_mult2):")
print(m_elem_mult1.element_mult(m_elem_mult2))


--- Demo: scalarmul(c) ---
Original Matrix m_arith:
1 2
3 4

m_arith.scalarmul(2):
2 4
6 8


--- Demo: add(N) ---
Matrix m_add1:
1 1
1 1

Matrix m_add2:
2 2
2 2

m_add1.add(m_add2):
3 3
3 3
--- Demo: sub(N) ---
Matrix m_sub1:
5 6
7 8

Matrix m_sub2:
1 1
1 1

m_sub1.sub(m_sub2):
4 5
6 7
--- Demo: mat_mult(N) (Matrix Multiplication) ---
Matrix m_mult_A:
1 2
3 4

Matrix m_mult_B:
5 6
7 8

m_mult_A.mat_mult(m_mult_B):
19 22
43 50

Matrix m_mult_C:
1 2 3

Matrix m_mult_D:
4
5
6

m_mult_C.mat_mult(m_mult_D):
32
--- Demo: element_mult(N) ---
Matrix m_elem_mult1:
1 2
3 4

Matrix m_elem_mult2:
10 20
30 40

m_elem_mult1.element_mult(m_elem_mult2):
10 40
90 160


problem 5 tests

In [14]:
print("--- Demo: Operator Overloading ---")

m1 = Matrix([[1, 2], [3, 4]])
m2 = Matrix([[5, 6], [7, 8]])
m_scalar = 3

print("Matrix m1:\n", m1)
print("Matrix m2:\n", m2)
print("Scalar:\n", m_scalar)

print("\n--- m1 + m2 (Matrix Addition) ---")
try:
    m_add_result = m1 + m2
    print(m_add_result)
except ValueError as e:
    print(f"Error: {e}")


print("\n--- m1 - m2 (Matrix Subtraction) ---")
try:
    m_sub_result = m1 - m2
    print(m_sub_result)
except ValueError as e:
    print(f"Error: {e}")

print("\n--- m1 * m_scalar (Scalar Multiplication, M * c) ---")
m_mul_scalar_right = m1 * m_scalar
print(m_mul_scalar_right)

print("\n--- m_scalar * m1 (Scalar Multiplication, c * M) ---")
m_mul_scalar_left = m_scalar * m1
print(m_mul_scalar_left)

print("\n--- m1 * m2 (Element-wise Multiplication) ---")
try:
    m_elem_mul_result = m1 * m2
    print(m_elem_mul_result)
except ValueError as e:
    print(f"Error: {e}")


print("\n--- m1 @ m2 (Matrix Multiplication) ---")
try:
    m_mat_mul_result = m1 @ m2
    print(m_mat_mul_result)
except ValueError as e:
    print(f"Error: {e}")

print("\n--- M == N (Equality Check) ---")
m_eq1 = Matrix([[1, 2], [3, 4]])
m_eq2 = Matrix([[1, 2], [3, 4]])
m_eq3 = Matrix([[5, 6], [7, 8]])

print(f"m1 == m_eq1: {m1 == m_eq1}")
print(f"m1 == m_eq3: {m1 == m_eq3}")
print(f"m1 == Matrix(2,3): {m1 == Matrix(2,3)}")


--- Demo: Operator Overloading ---
Matrix m1:
 1 2
3 4
Matrix m2:
 5 6
7 8
Scalar:
 3

--- m1 + m2 (Matrix Addition) ---
6 8
10 12

--- m1 - m2 (Matrix Subtraction) ---
-4 -4
-4 -4

--- m1 * m_scalar (Scalar Multiplication, M * c) ---
3 6
9 12

--- m_scalar * m1 (Scalar Multiplication, c * M) ---
3 6
9 12

--- m1 * m2 (Element-wise Multiplication) ---
5 12
21 32

--- m1 @ m2 (Matrix Multiplication) ---
19 22
43 50

--- M == N (Equality Check) ---
m1 == m_eq1: True
m1 == m_eq3: False
m1 == Matrix(2,3): False


problem 6 tests

In [16]:
print("--- Basic Matrix Properties Demonstration ---")

A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])
C = Matrix([[9, 10], [11, 12]])
I = eye(2)

print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Matrix C:\n", C)
print("Identity Matrix I:\n", I)

print("\n--- 1. Associative Property of Multiplication: (AB)C = A(BC) ---")
AB = A @ B
ABC_left = AB @ C
print("(AB)C:\n", ABC_left)

BC = B @ C
ABC_right = A @ BC
print("A(BC):\n", ABC_right)

print(f"Result: (AB)C == A(BC) is {ABC_left == ABC_right}")

print("\n--- 2. Distributive Property: A(B+C) = AB+AC ---")
B_plus_C = B + C
A_B_plus_C = A @ B_plus_C
print("A(B+C):\n", A_B_plus_C)

AB_dist = A @ B
AC_dist = A @ C
AB_plus_AC = AB_dist + AC_dist
print("AB + AC:\n", AB_plus_AC)

print(f"Result: A(B+C) == AB+AC is {A_B_plus_C == AB_plus_AC}")

print("\n--- 3. Non-Commutativity of Multiplication: AB != BA ---")
AB_non_commute = A @ B
BA_non_commute = B @ A
print("AB:\n", AB_non_commute)
print("BA:\n", BA_non_commute)

print(f"Result: AB == BA is {AB_non_commute == BA_non_commute} (Expected False)")

print("\n--- 4. Identity Property: AI = A ---")
AI_result = A @ I
print("AI:\n", AI_result)

print(f"Result: AI == A is {AI_result == A}")


--- Basic Matrix Properties Demonstration ---
Matrix A:
 1 2
3 4
Matrix B:
 5 6
7 8
Matrix C:
 9 10
11 12
Identity Matrix I:
 1.0 0.0
0.0 1.0

--- 1. Associative Property of Multiplication: (AB)C = A(BC) ---
(AB)C:
 413 454
937 1030
A(BC):
 413 454
937 1030
Result: (AB)C == A(BC) is True

--- 2. Distributive Property: A(B+C) = AB+AC ---
A(B+C):
 50 56
114 128
AB + AC:
 50 56
114 128
Result: A(B+C) == AB+AC is True

--- 3. Non-Commutativity of Multiplication: AB != BA ---
AB:
 19 22
43 50
BA:
 23 34
31 46
Result: AB == BA is False (Expected False)

--- 4. Identity Property: AI = A ---
AI:
 1.0 2.0
3.0 4.0
Result: AI == A is True
